# 18h — Corridors-out isolation test (PF-4, study plan v0.18.1 post-freeze register): the arm

**Question (register, verbatim):** does climate corridors influence the registered balanced map at all? This notebook writes the
arm for ONE certified anchor solve on the registered balanced formulation (S0 at SSP585, identity shapes, manifest v3.1 targets):
climate corridors removed from the objective (weight 0) and the connectivity block's full 25% share carried by transboundary
connectivity alone; weights re-derived under constant intended influence for the other blocks (`leverage_core.scenario_weights`
with a four-block structure whose connectivity block has one member). Zero solves, seconds. Pre-condition on Decision 1 (block
structure: B = transboundary alone vs the registered 50/50 block), which is HELD until the test returns.

**Run order:** 18h (this, the arm → `spec/v3.1/pf4_corridors_out.json`) → **18i** (R: the anchor, ~1 min) → **18j** (the comparison
and the register's reading rule).


In [1]:
# ---- bootstrap + the arm: corridors out, transboundary alone on the connectivity block's 25% ------------------------------
import importlib, json, pathlib, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
ROOT = _cands[0]; sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc
for _m in (config, lc, ec, dc):
    importlib.reload(_m)
VP = dc.VP; assert VP.version == "v3.1", "PF-4 is a post-freeze sensitivity on the v3.1 record"
G = dc.grid(); RUNS, REC, SPEC, PKG = dc.RUNS, dc.SPEC_REC, dc.SPEC, dc.PKG
FIG = ROOT / "analyses" / "y2y" / "figures" / VP.version; FIG.mkdir(exist_ok=True, parents=True)
FID, ARM = "s0_ssp585_theta5", "pf4_corridors_out"
sc = json.loads((SPEC / "scenarios_v2.json").read_text()); S0 = sc["S0_balanced"]
MAN = dc.package_manifest(pd.read_csv(dc.MANIFEST)); row = MAN[MAN.formulation_id == FID].iloc[0]
w_reg = json.loads(row.weight_vector); t_full = json.loads(row.target_vector)          # manifest v3.1 targets: m_soc 0.332 + 20 rarity-scaled EFG targets
wts = lambda **kw: {r.feature: round(float(r.w), 6) for r in lc.scenario_weights(S0["block_shares"], within_block=S0["within_block"], targets=S0["targets"], **kw).itertuples()}
w_chk = wts(); assert all(abs(w_chk[f] - w_reg[f]) < 1e-5 for f in w_reg), "re-derived S0 weights do not reproduce the manifest row"
# the corridors-out block structure: the connectivity block keeps its 25% share with ONE member, so transboundary's intended
# per-layer share goes 0.125 -> 0.25; the other three blocks keep their shares and within-block splits (carbon's 0.742 / 0.258)
BLK = {"core_habitat": ["climate_type_macrorefugia"], "connectivity": ["transboundary_connectivity"],
       "carbon": ["irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass"], "biodiversity": ["aoh_richness_birds", "aoh_richness_mammals"]}
w_pf4 = wts(blocks=BLK); w_pf4["climate_corridors"] = 0.0                              # removed from the objective
print(f"{'feature':32s} {'registered':>10s} {'PF-4':>10s}   intended per-layer share (registered -> PF-4)")
share_reg = {f: S0["block_shares"][b] * (S0["within_block"].get(b, {}).get(f, 1.0 / len(m))) for b, m in config.BLOCKS.items() for f in m}
share_new = {f: S0["block_shares"][b] * (S0["within_block"].get(b, {}).get(f, 1.0 / len(m))) for b, m in BLK.items() for f in m}
for f in w_reg:
    print(f"{f:32s} {w_reg[f]:10.4f} {w_pf4[f]:10.4f}   {share_reg[f]:.3f} -> {share_new.get(f, 0.0):.3f}")
payload = dict(created_utc=datetime.now(timezone.utc).isoformat(), spec="study plan v0.18.1 post-freeze register PF-4 (corridors-out isolation test)",
    base_formulation=FID, arm=ARM, label="PF-4: corridors out -- climate corridors removed from the objective, transboundary connectivity alone on the connectivity block's 25%",
    blocks=BLK, block_shares=S0["block_shares"], within_block=S0["within_block"], intended_share_registered=share_reg, intended_share_pf4=share_new,
    weights=w_pf4, weights_registered=w_reg, targets=t_full, opt_gap=float(row.opt_gap),
    normalization="derived weights rescaled to mean 1 over the six blocked features (scenario_weights default); outside features at their baselines (naturalness w = 1, EFGs at the engine's 1/n)",
    out_rel=f"analyses/y2y/runs_v3.1/{ARM}", reference_registered=f"analyses/y2y/runs_v3.1/{FID}")
(REC / "pf4_corridors_out.json").write_text(json.dumps(payload, indent=2))
print(f"\nwrote {REC.relative_to(ROOT) / 'pf4_corridors_out.json'} | opt_gap {payload['opt_gap']:g} -> next: 18i (the anchor), then 18j (the comparison)")


feature                          registered       PF-4   intended per-layer share (registered -> PF-4)
climate_type_macrorefugia            1.4600     1.3481   0.250 -> 0.250
transboundary_connectivity           0.6693     1.2361   0.125 -> 0.250
climate_corridors                    1.1714     0.0000   0.125 -> 0.000
irrecoverable_carbon_m_soc           0.4646     0.4290   0.185 -> 0.185
irrecoverable_carbon_biomass         0.1986     0.1833   0.065 -> 0.065
aoh_richness_birds                   1.3286     1.2268   0.125 -> 0.125
aoh_richness_mammals                 1.7075     1.5767   0.125 -> 0.125

wrote analyses/y2y/spec/v3.1/pf4_corridors_out.json | opt_gap 0.0001 -> next: 18i (the anchor), then 18j (the comparison)
